# 1. Single Prompt Analysis

The core demonstration: tracing probability displacement across alignment layers for a single prompt.

In [ ]:
import pandas as pd
import numpy as np
from plotnine import *
from malign_logits.psyche import Psyche
from pathlib import Path

FIGDIR = Path.home() / 'Dropbox/Prof/Confs/Accelerationism/figures'
FIGDIR.mkdir(parents=True, exist_ok=True)

theme_malign = theme_minimal() + theme(
    figure_size=(10, 6),
    plot_background=element_rect(fill='white'),
    panel_grid_minor=element_blank(),
    text=element_text(family='serif'),
    plot_title=element_text(size=14, weight='bold'),
    plot_subtitle=element_text(size=11, color='#666'),
)

In [ ]:
# Load OLMo (4 layers) — uses cached logits if available
psyche = Psyche.from_family("olmo", load=True)

prompt = "She was so angry she wanted to"
analysis = psyche.analyze(prompt)
analysis.formation_report()

## Formation trajectories

Each line traces a word's probability across training stages. Log scale — small absolute changes represent large relative shifts.

In [ ]:
df = analysis.formation_df.copy()

layer_cols = [c for c in ['base', 'ego', 'superego', 'instruct'] if c in df.columns]
layer_labels = {'base': 'Base (Id)', 'ego': 'SFT (Ego)', 'superego': 'DPO (Superego)', 'instruct': 'RLVR (Ego-ideal)'}

for i in range(len(layer_cols) - 1):
    df[f'_delta_{i}'] = (df[layer_cols[i]] - df[layer_cols[i+1]]).abs()
delta_cols = [c for c in df.columns if c.startswith('_delta_')]
df['max_delta'] = df[delta_cols].max(axis=1)
top = df.nlargest(30, 'max_delta')

long = top.melt(id_vars=['word', 'trajectory'], value_vars=layer_cols,
                var_name='layer', value_name='probability')
long['layer'] = pd.Categorical(long['layer'], categories=layer_cols, ordered=True)
long['layer_label'] = long['layer'].map(layer_labels)
long['layer_label'] = pd.Categorical(long['layer_label'], 
                                      categories=[layer_labels[c] for c in layer_cols], 
                                      ordered=True)
long['probability'] = long['probability'].clip(lower=1e-7)

p = (ggplot(long, aes(x='layer_label', y='probability', group='word', color='trajectory'))
 + geom_line(alpha=0.6, size=0.8)
 + geom_point(size=1.5, alpha=0.7)
 + geom_text(aes(label='word'), 
             data=long[long.layer == layer_cols[-1]],
             nudge_x=0.15, size=7, ha='left', alpha=0.8)
 + scale_y_log10()
 + scale_color_manual(values={
     'decline': '#e15759', 'rise': '#4e79a7', 'V': '#f28e2b',
     'peak': '#76b7b2', 'sublimated': '#b07aa1', 'eliminated': '#b07aa1',
     'superego_only': '#59a14f', 'flat': '#9c9c9c'
 })
 + labs(title=f'Formation trajectories: "{prompt}"',
        subtitle='Top 30 words by maximum displacement between adjacent layers',
        x='', y='Probability (log scale)', color='Trajectory')
 + theme_malign
 + theme(figure_size=(12, 8))
)
p.save(FIGDIR / '01_formation_trajectories.png', dpi=200)
p